In [ ]:
!pip install pandas torch wfdb torchmetrics

In [ ]:
import pandas as pd
import numpy as np
import wfdb
import ast
import torch
from sklearn.preprocessing import MultiLabelBinarizer
import torchmetrics
import os

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [3]:
def load_raw_data(df, sampling_rate):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(os.path.join(os.curdir, f)) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(os.path.join(os.curdir, f)) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data

sampling_rate=100

# load and convert annotation data
Y = pd.read_csv(os.path.join(os.curdir,'ptbxl_database.csv'), index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))

# Load raw signal data
X = load_raw_data(Y, sampling_rate)

# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(os.path.join(os.curdir,'scp_statements.csv'), index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

# Apply diagnostic superclass
Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)

In [4]:
# Split data into train and test
test_fold = 10
# Train
X_train = X[np.where(Y.strat_fold != test_fold)]
y_train = Y[(Y.strat_fold != test_fold)].diagnostic_superclass
# Test
X_test = X[np.where(Y.strat_fold == test_fold)]
y_test = Y[Y.strat_fold == test_fold].diagnostic_superclass

---End of code provided by ECG dataset---

In [17]:
mlb = MultiLabelBinarizer()

y_train = mlb.fit_transform(y_train)
y_test = mlb.transform(y_test)

In [45]:
X_train.shape, X_train.dtype, y_train.shape, y_train.dtype

((19601, 1000, 12), dtype('float64'), (19601, 5), dtype('int64'))

In [47]:
summed = np.sum(y_train, axis=1)
record_ids_with_no_labels_true = []
for id, class_sum in enumerate(summed):
    if class_sum == 0:
        record_ids_with_no_labels_true.append(id)
print(f'{len(record_ids_with_no_labels_true)} train examples have all labels as "0"')

371 train examples have all labels as "0"


In [48]:
X_train_tensor = torch.tensor(X_train,dtype=torch.float32).transpose(1,2)  # the 12 channels dimension is initially loaded in as last
y_train_tensor = torch.tensor(y_train,dtype=torch.float32)

X_test_tensor = torch.tensor(X_test,dtype=torch.float32).transpose(1,2)
y_test_tensor = torch.tensor(y_test,dtype=torch.float32)

In [49]:
from torch.utils.data import TensorDataset,DataLoader
train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.conv1 = nn.Conv1d(12, 32, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(2)

        self.conv3 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(2)

        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):

        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))

        x = self.global_pool(x)
        x = x.squeeze(-1)

        x = self.fc(x)

        return x

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()

In [ ]:
model = CNN(num_classes=y_train_tensor.shape[1]).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_loss = float('inf')  # start with infinity
save_path = "best_model.pt"

for epoch in range(10):
    running_loss = 0.0
    model.train()

    for x, y in train_loader:

        x = x.to(device)
        y = y.to(device)

        preds = model(x)

        loss = loss_fn(preds, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}")
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), save_path)

In [ ]:
import torch

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()

        all_preds.append(preds.cpu())
        all_labels.append(y.cpu())

# Concatenate all batches
all_preds = torch.cat(all_preds, dim=0)
all_labels = torch.cat(all_labels, dim=0)

In [ ]:
# Per-class accuracy
per_class_acc = (all_preds == all_labels).float().mean(dim=0)  # [num_classes]
# Print results
for idx, acc in enumerate(per_class_acc):
    print(f"Class {mlb.classes_[idx]} accuracy: {acc.item():.3f}")

In [ ]:
from sklearn.metrics import f1_score
f1_scores = f1_score(all_labels, all_preds, average=None)
for idx, score in enumerate(f1_scores):
    print(f"Class {mlb.classes_[idx]} F1-score: {score:.3f}")

In [ ]:
from sklearn.metrics import roc_auc_score

num_classes = all_labels.shape[1]
aucs = []

for i in range(num_classes):
    try:
        auc = roc_auc_score(all_labels[:, i], all_preds[:, i])
    except ValueError:
        # happens if class i has no positive samples in test set
        auc = float('nan')
    aucs.append(auc)

for idx, auc in enumerate(aucs):
    print(f"Class {mlb.classes_[idx]} AUROC: {auc:.3f}")